# Citations & Attribution: make every claim checkable

A RAG answer can *sound* authoritative and still be unverifiable. Did the model read a claim off the
retrieved passages, or invent it? Without citations a fluent **wrong** answer is indistinguishable
from a right one. This notebook makes each claim **checkable**: it decomposes an answer into claims,
**attributes** each claim to the retrieved passage that best supports it, assigns an inline citation,
and **measures** the result (citation precision & recall) — while flagging any claim with no support
as **UNCITABLE**, the likely hallucination.

Everything runs on **CPU** in a few seconds. We import the *same* canonical functions the concept
page and the `.py` use (`citations_attribution.py`, which reuses ch5's `DenseRetriever`), so every
number here is the chapter's own — asserted before it is claimed.

**What is real vs illustrative** (stated up front, and again in the honesty banner the first cell
prints):
- **Real & measured:** the attribution matching (each claim's cosine to each retrieved passage via
  ch5's `all-MiniLM` encoder — the same one ch8/ch11 use), the citation assignment, and citation
  precision/recall.
- **Illustrative (labelled):** claim decomposition (a sentence splitter stands in for an LLM
  extractor) and generation-time citation (no generator here — we point at the real APIs).
- **Carried caveat:** cosine ≈ *topical* similarity, **not** entailment — so a topically-near
  passage can earn a **false citation**. We show that explicitly, and it is why ALCE/AIS use an NLI
  entailment model, which cosine only approximates.


> **Step 1 — import the canonical code + print the reproducibility banner.** Everything comes
> from `citations_attribution.py`; nothing is re-defined here. The banner prints `numpy`/`torch`
> versions and the detected accelerator so the run is reproducible (the encoder is CPU-pinned).

In [1]:
import numpy as np
import torch

from citations_attribution import (
    ANSWER,
    FALSE_CITE_CLAIM,
    QUESTION,
    SUPPORT_THRESHOLD,
    DenseRetriever,
    attribute_claims,
    build_golds,
    citation_precision,
    citation_recall,
    claim_passage_scores,
    full_corpus,
    render_cited_answer,
    retrieve_passages,
    split_into_claims,
)

print("numpy:", np.__version__)
_acc = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("torch:", torch.__version__, "| accelerator available:", _acc, "| encoder runs on: cpu")

corpus = full_corpus()
dense = DenseRetriever(corpus)
passages, top_idx = retrieve_passages(dense, corpus, k=3)
print(f"corpus: {len(corpus)} passages | dense lens: {dense.backend} | support threshold: {SUPPORT_THRESHOLD}")
print("NOTE: attribution matching + citation precision/recall are REAL; the claim-splitter and the generator are illustrative LLM stand-ins.")

numpy: 2.4.6
torch: 2.12.0 | accelerator available: mps | encoder runs on: cpu


corpus: 11 passages | dense lens: sentence-transformers/all-MiniLM-L6-v2 | support threshold: 0.5
NOTE: attribution matching + citation precision/recall are REAL; the claim-splitter and the generator are illustrative LLM stand-ins.


> **Step 2 — the question, the retrieved context, and the answer we will attribute.** The answer
> reads fluently and mixes **two grounded claims** with **one hallucination** ("powered entirely by
> solar panels" — a fact no passage states). That is exactly the case citations exist to catch.

In [2]:
print("QUESTION:", QUESTION)
print()
print(f"retrieved context (top-{len(passages)}, corpus ids {list(top_idx)}):")
for n, p in enumerate(passages, start=1):
    print(f"  [{n}] doc[{top_idx[n-1]}]: {p}")
print()
print("ANSWER (raw):", ANSWER)

QUESTION: When did Helios-7 launch, what is its imager resolution, and what powers it?

retrieved context (top-3, corpus ids [1, 0, 2]):
  [1] doc[1]: Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
  [2] doc[0]: The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.
  [3] doc[2]: The project lead for Helios-7 is Dr. Amara Okoye, based in the Nairobi office.

ANSWER (raw): Helios-7 launched on March 3rd, 2024 from the Kourou spaceport. Its hyperspectral imager has a ground resolution of 4 meters. Helios-7 is powered entirely by solar panels.


## 1) Decompose → attribute → assign / flag

The whole mechanism, from primitives: split the answer into claims, score each claim against every
retrieved passage (real encoder cosine), and cite the best passage **iff** its cosine clears the
support threshold τ = 0.5 — otherwise flag the claim **UNCITABLE**.

> **Step 3 — split the answer into claims.** An illustrative sentence splitter stands in for an
> LLM claim extractor (labelled); the per-claim attribution that follows is real.

In [3]:
claims = split_into_claims(ANSWER)
for i, c in enumerate(claims, start=1):
    print(f"claim {i}: {c}")

claim 1: Helios-7 launched on March 3rd, 2024 from the Kourou spaceport.
claim 2: Its hyperspectral imager has a ground resolution of 4 meters.
claim 3: Helios-7 is powered entirely by solar panels.


> **Step 4 — attribute each claim and read the per-claim verdict.** Each grounded claim cites the
> passage that entails it; the fabricated claim clears no passage and is flagged UNCITABLE.

In [4]:
attributions = attribute_claims(dense, ANSWER, passages)
for i, a in enumerate(attributions, start=1):
    if a.citation is not None:
        print(f"claim {i}: CITED [{a.citation}] (cos {a.best_score:.3f})  <- {passages[a.citation-1]}")
    else:
        print(f"claim {i}: UNCITABLE (best cos {a.best_score:.3f} < {SUPPORT_THRESHOLD})  <- no passage supports this")
print()
print("ANSWER (with citations):", render_cited_answer(attributions))

claim 1: CITED [2] (cos 0.933)  <- The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.
claim 2: CITED [1] (cos 0.782)  <- Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
claim 3: UNCITABLE (best cos 0.488 < 0.5)  <- no passage supports this

ANSWER (with citations): Helios-7 launched on March 3rd, 2024 from the Kourou spaceport. [2] Its hyperspectral imager has a ground resolution of 4 meters. [1] Helios-7 is powered entirely by solar panels. [?]


> **Step 5 — assert the contrast before believing it.** Two claims cite a real passage; the
> solar-panels hallucination is UNCITABLE. This is the headline: the cited claims check out, the
> fabricated one cannot hide.

In [5]:
n_cited = sum(1 for a in attributions if a.citation is not None)
assert n_cited == 2, "exactly the two grounded claims should be cited"
assert attributions[0].citation is not None and attributions[1].citation is not None, "both grounded claims cite a passage"
assert attributions[-1].citation is None, "the solar-panels hallucination must be UNCITABLE"
print(f"{n_cited}/3 claims cited to a real passage; the hallucination is flagged [?] — asserted True.")

2/3 claims cited to a real passage; the hallucination is flagged [?] — asserted True.


![Every claim attached to its source: the two grounded claims draw a green line to the passage
they cite (with the real cosine), and the fabricated claim — matching no passage — is flagged
UNCITABLE in red. Generated by `make_figures_13.py` from this same attribution output.](../../images/rag13_cited_answer.png)

## 2) Measure it: citation precision & recall

Attribution is only trustworthy if we can **grade** it. We supply a labelled gold (which passages
*actually* support each claim — a stand-in for a human/NLI judge) and score the attributor:
- **citation precision** — of the claims that cite a source, how many point to a genuinely-supporting
  passage? (punishes false citations & over-citation)
- **citation recall** — of the supportable claims, how many got a correct citation? (punishes missing
  citations). A hallucinated claim is **not** in recall's denominator — correctly leaving it uncited
  helps precision, not recall.

> **Step 6 — score the attributor against the gold.** On this clean case it is perfect on both
> axes: no false citation (precision 1.0), no missed citation (recall 1.0).

In [6]:
golds = build_golds(passages)
precision = citation_precision(attributions, golds)
recall = citation_recall(attributions, golds)
print(f"citation precision = {precision:.3f}  (of emitted citations, how many point to a supporting passage)")
print(f"citation recall    = {recall:.3f}  (of supportable claims, how many got a correct citation)")
assert precision == 1.0, "every emitted citation points to a genuinely-supporting passage"
assert recall == 1.0, "both supportable claims received a correct citation"
print("asserted: precision 1.0, recall 1.0 on the clean case.")

citation precision = 1.000  (of emitted citations, how many point to a supporting passage)
citation recall    = 1.000  (of supportable claims, how many got a correct citation)
asserted: precision 1.0, recall 1.0 on the clean case.


![The claim × passage cosine matrix. Each row is a claim; the argmax cell (the citation) is
boxed — green when it clears τ, red-dashed when the argmax is *still* below τ (the hallucination's
row: uncitable). Generated by `make_figures_13.py`.](../../images/rag13_attribution_heatmap.png)

## 3) The pitfall: cosine ≈ topic, not entailment → a FALSE citation

The honest limit carried from ch8/ch11: encoder cosine measures *topical* similarity, not factual
**entailment**. A claim topically near a passage it does **not** entail (indeed, one the passage
*contradicts*) still scores high — so the matcher assigns a **false citation**. This is exactly why
ALCE (Gao et al. 2023) and AIS (Rashkin et al. 2021 / Bohnet et al. 2022) define citation quality
with an **NLI entailment model**, not cosine.

> **Step 7 — a wrong-lead claim cites the project-lead passage it contradicts.** The claim names
> a *different* lead in a *different* city, yet it shares the passage's vocabulary — so cosine rates
> it above the bar and cites it. A citation that is topically plausible but factually wrong.

In [7]:
scores = claim_passage_scores(dense, FALSE_CITE_CLAIM, passages)
best = int(np.argmax(scores))
lead_num = next(i for i, p in enumerate(passages) if "project lead" in p)
print("claim (CONTRADICTED by the passage it will be cited to):", FALSE_CITE_CLAIM)
print(f"cosine cites passage [{best+1}] at cos {scores[best]:.3f}  (clears {SUPPORT_THRESHOLD}: {scores[best] >= SUPPORT_THRESHOLD})")
print(f"  passage [{best+1}]: {passages[best]}")
assert scores[best] >= SUPPORT_THRESHOLD, "the wrong-lead claim clears the cosine bar despite NOT being entailed"
assert best == lead_num, "cosine wrongly cites the topically-near project-lead passage"
print("\nasserted: a false citation cosine cannot catch — 'about the lead' != 'entails THIS lead'. NLI is the fix.")

claim (CONTRADICTED by the passage it will be cited to): The Helios-7 project lead is Dr. Lars Vinter, based in the Oslo office.
cosine cites passage [3] at cos 0.711  (clears 0.5: True)
  passage [3]: The project lead for Helios-7 is Dr. Amara Okoye, based in the Nairobi office.

asserted: a false citation cosine cannot catch — 'about the lead' != 'entails THIS lead'. NLI is the fix.


## 4) Over-citation drops precision

Turn the support threshold **off** (τ = 0) and the attributor is forced to cite *every* claim,
including the hallucination. That extra citation is **false**, so precision falls; recall (which
counts only supportable claims) is unchanged. The threshold is the precision/recall dial.

> **Step 8 — force a citation on every claim and watch precision fall.** The hallucination gets
> force-cited to its (wrong) best passage; precision drops from 1.00 to 0.67, recall holds at 1.00.

In [8]:
greedy = attribute_claims(dense, ANSWER, passages, threshold=0.0)  # cite even the hallucination
p_over = citation_precision(greedy, golds)
r_over = citation_recall(greedy, golds)
forced = greedy[-1]
print(f"with threshold 0, the hallucination is force-cited to passage [{forced.citation}] (cos {forced.best_score:.3f})")
print(f"citation precision: {precision:.3f} (thresholded)  ->  {p_over:.3f} (over-cited)")
print(f"citation recall   : {recall:.3f} (thresholded)  ->  {r_over:.3f} (unchanged)")
assert p_over < precision, "force-citing the hallucination adds a false citation -> precision drops"
assert r_over == recall, "over-citation does not change recall"
print("\nasserted: precision 1.00 -> 0.67, recall unchanged. Over-citing the hallucination is a false citation.")

with threshold 0, the hallucination is force-cited to passage [3] (cos 0.488)
citation precision: 1.000 (thresholded)  ->  0.667 (over-cited)
citation recall   : 1.000 (thresholded)  ->  1.000 (unchanged)

asserted: precision 1.00 -> 0.67, recall unchanged. Over-citing the hallucination is a false citation.


![Citation precision & recall: the clean thresholded attributor (green) vs an over-citing one
(red, τ=0). Over-citing the hallucination adds a false citation, so precision drops 1.00 → 0.67
while recall holds at 1.00. Generated by `make_figures_13.py`.](../../images/rag13_precision_recall.png)

## Try it yourself

Before you run the next cell, **predict**. So far every grounded claim happened to paraphrase its
source closely (cos 0.93, 0.78). Now we add a **second hallucination** — a claim about the launch
*site's weather* that no passage states — to the answer.

**Question:** what will the new answer's citation **precision** and **recall** be? Recall counts only
the *supportable* claims (still the launch + resolution claims → 2). Precision counts *emitted*
citations. Will the two new-hallucination be flagged UNCITABLE (so precision stays 1.0), or will one
sneak a false citation?

Think, then run.

In [9]:
two_hallucinations = (
    ANSWER  # the two grounded claims + the solar-panels hallucination
    + " The Kourou spaceport enjoyed clear skies on launch day."  # a SECOND hallucination
)
attr2 = attribute_claims(dense, two_hallucinations, passages)
# gold: launch + resolution supportable; BOTH the solar-panels and the weather claim are not
from citations_attribution import CitedClaimGold  # noqa: E402
launch_p = next(i + 1 for i, p in enumerate(passages) if "Kourou" in p)
imager_p = next(i + 1 for i, p in enumerate(passages) if "ground resolution" in p)
golds2 = (
    CitedClaimGold(True, frozenset({launch_p})),   # launch claim
    CitedClaimGold(True, frozenset({imager_p})),   # resolution claim
    CitedClaimGold(False, frozenset()),            # solar-panels hallucination
    CitedClaimGold(False, frozenset()),            # weather hallucination
)
for i, a in enumerate(attr2, start=1):
    tag = f"[{a.citation}]" if a.citation is not None else "[?] UNCITABLE"
    print(f"claim {i}: {tag:>14} (best cos {a.best_score:.3f})  {a.claim}")
p2 = citation_precision(attr2, golds2)
r2 = citation_recall(attr2, golds2)
print(f"\ncitation precision = {p2:.3f}   citation recall = {r2:.3f}")

claim 1:            [2] (best cos 0.933)  Helios-7 launched on March 3rd, 2024 from the Kourou spaceport.
claim 2:            [1] (best cos 0.782)  Its hyperspectral imager has a ground resolution of 4 meters.
claim 3:  [?] UNCITABLE (best cos 0.488)  Helios-7 is powered entirely by solar panels.
claim 4:            [2] (best cos 0.535)  The Kourou spaceport enjoyed clear skies on launch day.

citation precision = 0.667   citation recall = 1.000


> **Was your prediction right?** The weather claim shares the launch passage's *vocabulary*
> ("Kourou", "launch") — so watch whether cosine flags it or false-cites it. Whatever happened,
> the lesson is the same one the false-citation cell made: a topical match is **not** entailment.
> The next cell asserts the outcome so the notebook stays honest about it.

In [10]:
# Recall's denominator is the supportable claims (launch + resolution); both still cite correctly.
assert r2 == 1.0, "the two grounded claims still get correct citations -> recall stays 1.0"
# Precision depends on whether either hallucination sneaks a citation above tau.
n_emitted = sum(1 for a in attr2 if a.citation is not None)
n_false = sum(1 for a, g in zip(attr2, golds2) if a.citation is not None and a.citation not in g.supporting)
print(f"emitted citations: {n_emitted}   false citations: {n_false}   -> precision {p2:.3f}")
assert p2 <= precision, "adding a hallucination can only hold or lower precision, never raise it"
print("asserted: recall holds at 1.0; precision reflects exactly how many hallucinations sneaked a false cite.")

emitted citations: 3   false citations: 1   -> precision 0.667
asserted: recall holds at 1.0; precision reflects exactly how many hallucinations sneaked a false cite.


## What we saw

- **Attribution makes a claim checkable** — each claim is mapped back to the passage it came from, so
  a reader can verify it instead of trusting fluent prose. The two grounded claims cited real
  passages (cos 0.93, 0.78); the fabricated "solar panels" claim matched none and was flagged
  **UNCITABLE** (best cos 0.49 < 0.5).
- **Citation precision & recall grade the attributor** — 1.00 / 1.00 on the clean case; over-citing
  the hallucination (τ=0) dropped precision to **0.67** while recall held at 1.00. The threshold is
  the precision/recall dial.
- **Cosine ≈ topic, not entailment** — a wrong-lead claim earned a **false citation** to the
  project-lead passage it *contradicts* (cos 0.71). That gap is why ALCE/AIS score citations with an
  **NLI entailment model**; the cosine matcher only approximates it.
- **Two regimes:** we built **post-hoc** attribution (map claims back after generation — works on any
  output). **Generation-time** citation (the model emits [1][2] as it writes) is the library
  one-liner path — Anthropic's Citations API, Vertex AI grounding, LlamaIndex's `CitationQueryEngine`
  — covered on the concept page.

**Provenance:** the metric definitions follow **ALCE** (Gao et al. 2023, arXiv:2305.14627) and the
attribution framework **AIS** (Rashkin et al. 2021, arXiv:2112.12870; Bohnet et al. 2022,
arXiv:2212.08037) — all in the references file.
